In [ ]:
# Restart kernel first, then run this
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

PROCESSED = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning\data\processed"
RESULTS   = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning\results"
MODELS    = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning\models"

X_train = np.load(os.path.join(PROCESSED, "X_train.npy"))
X_val   = np.load(os.path.join(PROCESSED, "X_val.npy"))
X_test  = np.load(os.path.join(PROCESSED, "X_test.npy"))
y_train = np.load(os.path.join(PROCESSED, "y_train.npy"))
y_val   = np.load(os.path.join(PROCESSED, "y_val.npy"))
y_test  = np.load(os.path.join(PROCESSED, "y_test.npy"))
class_weights_arr = np.load(os.path.join(PROCESSED, "class_weights.npy"), allow_pickle=True)
class_weights = {0: float(class_weights_arr[0]), 1: float(class_weights_arr[1])}

# Reshape
X_train_rnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val_rnn   = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_rnn  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Shape: {X_train_rnn.shape}")

# ── Lighter BiLSTM ────────────────────────────────────────────────────────
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Input(shape=(50, 1)),
    keras.layers.Bidirectional(keras.layers.LSTM(32, return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(32)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_rnn, y_train,
    validation_data=(X_val_rnn, y_val),
    epochs=200,
    batch_size=2048,  # larger batch = less memory pressure
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1)

print(f"\nStopped at epoch: {early_stop.stopped_epoch if early_stop.stopped_epoch > 0 else 200}")